## test_builtin_claude

In [2]:
import json
import logging
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import requests
from bs4 import BeautifulSoup

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)

# ------------------------------------------------------------------ #
# Config
# ------------------------------------------------------------------ #
BASE_SEARCH_URL = (
    "https://www.builtinnyc.com/jobs/entry-level/junior/mid-level"
    "?search=Data+Analyst"
    "&daysSinceUpdated=30"
    "&city=New+York+City"
    "&state=New+York"
    "&country=USA"
    "&allLocations=true"
)

HEADERS = {
    "User-Agent":      "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/122.0.0.0 Safari/537.36",
    "Accept":          "text/html,application/xhtml+xml,application/xml;"
                       "q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer":         "https://www.builtinnyc.com/",
}

MAX_PAGES        = 10    # hard ceiling; early exit handles real stopping condition
MIN_CRAWL_DELAY  = 1.5   # seconds between index page requests
MAX_CRAWL_DELAY  = 3.5
MIN_SCRAPE_DELAY = 2.0   # seconds between individual job page requests
MAX_SCRAPE_DELAY = 4.5   # slightly longer — hitting more endpoints

CACHE_DIR = Path("./cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CRAWL_CACHE  = CACHE_DIR / "builtin_crawl_results.json"
SCRAPE_CACHE = CACHE_DIR / "builtin_scraped_jobs.json"

print("Config OK")
print(f"Crawl cache  : {CRAWL_CACHE.resolve()}")
print(f"Scrape cache : {SCRAPE_CACHE.resolve()}")

Config OK
Crawl cache  : /Users/vanbrantley/code/nyc-data-job-market-tracker/ingestion/notebooks/cache/builtin_crawl_results.json
Scrape cache : /Users/vanbrantley/code/nyc-data-job-market-tracker/ingestion/notebooks/cache/builtin_scraped_jobs.json


In [3]:
if CRAWL_CACHE.exists():
    print("Crawl cache exists — loading from file.")
    with CRAWL_CACHE.open() as f:
        discovered_jobs = json.load(f)
else:
    discovered_jobs: list[dict] = []
    seen_urls: set[str] = set()

    log.info("Starting Stage 1 crawl...")

    for page_num in range(1, MAX_PAGES + 1):
        url = BASE_SEARCH_URL if page_num == 1 else f"{BASE_SEARCH_URL}&page={page_num}"

        if page_num > 1:
            delay = random.uniform(MIN_CRAWL_DELAY, MAX_CRAWL_DELAY)
            log.info(f"Sleeping {delay:.2f}s before page {page_num}...")
            time.sleep(delay)

        log.info(f"Fetching index page {page_num}: {url}")

        try:
            response = requests.get(url, headers=HEADERS, timeout=15)

            if response.status_code != 200:
                log.warning(f"Page {page_num} returned HTTP {response.status_code} — skipping.")
                continue

            soup       = BeautifulSoup(response.text, "html.parser")
            script_tag = soup.find("script", type="application/ld+json")

            # Early exit: no JSON-LD block means we've gone past the last real page
            if not script_tag:
                log.info(f"No JSON-LD found on page {page_num} — end of results.")
                break

            payload    = json.loads(script_tag.string)
            graph      = payload.get("@graph", [])
            item_list  = next(
                (item for item in graph if item.get("@type") == "ItemList"),
                None
            )

            # Secondary exit: JSON-LD exists but job list is empty
            if not item_list or not item_list.get("itemListElement"):
                log.info(f"Empty ItemList on page {page_num} — end of results.")
                break

            elements  = item_list["itemListElement"]
            new_count = 0

            for el in elements:
                job_url   = el.get("url")
                job_title = el.get("name")

                if job_url and job_url not in seen_urls:
                    seen_urls.add(job_url)
                    discovered_jobs.append({
                        "title": job_title,
                        "url":   job_url,
                    })
                    new_count += 1

            log.info(f"Page {page_num}: {new_count} new jobs "
                     f"(running total: {len(discovered_jobs)})")

        except Exception as e:
            log.error(f"Error on page {page_num}: {e}", exc_info=True)

    with CRAWL_CACHE.open("w") as f:
        json.dump(discovered_jobs, f, indent=2)
    log.info(f"Crawl complete. {len(discovered_jobs)} jobs saved to {CRAWL_CACHE}")

print(f"\nTotal jobs discovered: {len(discovered_jobs)}")
print()
print(f"{'#':<4} {'Title':<60} {'URL'}")
print("-" * 120)
for i, job in enumerate(discovered_jobs):
    print(f"{i:<4} {job['title']:<60} {job['url']}")

2026-05-24 17:52:10,696 [INFO] Starting Stage 1 crawl...
2026-05-24 17:52:10,702 [INFO] Fetching index page 1: https://www.builtinnyc.com/jobs/entry-level/junior/mid-level?search=Data+Analyst&daysSinceUpdated=30&city=New+York+City&state=New+York&country=USA&allLocations=true
2026-05-24 17:52:11,378 [INFO] Page 1: 25 new jobs (running total: 25)
2026-05-24 17:52:11,378 [INFO] Sleeping 1.96s before page 2...
2026-05-24 17:52:13,348 [INFO] Fetching index page 2: https://www.builtinnyc.com/jobs/entry-level/junior/mid-level?search=Data+Analyst&daysSinceUpdated=30&city=New+York+City&state=New+York&country=USA&allLocations=true&page=2
2026-05-24 17:52:13,914 [INFO] Page 2: 21 new jobs (running total: 46)
2026-05-24 17:52:13,915 [INFO] Sleeping 2.32s before page 3...
2026-05-24 17:52:16,238 [INFO] Fetching index page 3: https://www.builtinnyc.com/jobs/entry-level/junior/mid-level?search=Data+Analyst&daysSinceUpdated=30&city=New+York+City&state=New+York&country=USA&allLocations=true&page=3
2026


Total jobs discovered: 46

#    Title                                                        URL
------------------------------------------------------------------------------------------------------------------------
0    Data Analyst, Data Cloud Intelligence                        https://www.builtinnyc.com/job/data-analyst-data-cloud-intelligence/9422585
1    Business Intelligence Data Analyst                           https://www.builtinnyc.com/job/business-intelligence-data-analyst/9246907
2    Data Governance Analyst                                      https://www.builtinnyc.com/job/data-governance-analyst/9224839
3    Private Markets Data Analyst/Senior Associate                https://www.builtinnyc.com/job/private-markets-data-analyst-senior-associate/9177772
4    Data Analyst                                                 https://www.builtinnyc.com/job/data-analyst/9483033
5    Data Analyst - Investment products (CONTRACT)                https://www.builtinnyc.com/job/data

In [4]:
# Pick one URL from the crawl results to inspect
sample_url = discovered_jobs[3]["url"]  # change index to inspect different jobs
print(f"Inspecting: {sample_url}\n")

response = requests.get(sample_url, headers=HEADERS, timeout=15)
print(f"HTTP status: {response.status_code}")

soup       = BeautifulSoup(response.text, "html.parser")
script_tag = soup.find("script", type="application/ld+json")

if not script_tag:
    print("❌ No JSON-LD found on this page.")
else:
    raw = json.loads(script_tag.string)
    graph = raw.get("@graph", [])

    print(f"@graph node types: {[node.get('@type') for node in graph]}\n")

    # Find the JobPosting node specifically
    job_posting = next(
        (node for node in graph if node.get("@type") == "JobPosting"),
        None
    )

    if not job_posting:
        print("❌ No JobPosting node found in @graph.")
    else:
        print("=== JobPosting node — all fields ===\n")
        for k, v in job_posting.items():
            display = v
            if isinstance(v, str) and len(v) > 120:
                display = v[:120] + " [...]"
            elif isinstance(v, dict):
                display = f"[dict] {list(v.keys())}"
            print(f"  {k:<35} {repr(display)}")

Inspecting: https://www.builtinnyc.com/job/private-markets-data-analyst-senior-associate/9177772

HTTP status: 200
@graph node types: ['JobPosting']

=== JobPosting node — all fields ===

  @context                            'https://schema.org'
  @type                               'JobPosting'
  title                               'Private Markets Data Analyst/Senior Associate'
  description                         '<p><b>Position Overview</b>:</p><p>The Data Strategy team within Neuberger Private Markets provides centralized oversigh [...]'
  directApply                         False
  identifier                          "[dict] ['@type', 'name', 'value']"
  baseSalary                          "[dict] ['@type', 'currency', 'value']"
  datePosted                          '2026-05-24'
  employmentType                      'FULL_TIME'
  applicantLocationRequirements       "[dict] ['@type', 'name']"
  hiringOrganization                  "[dict] ['@type', 'name', 'sameAs', 'logo']"
  in

In [6]:
# Tests field presence/consistency across a sample of jobs
# before committing to the full scrape in Cell 4

sample_indices = [0, 5, 10, 20, 30, 40, 45]  # spread across the full list
results = []

for idx in sample_indices:
    if idx >= len(discovered_jobs):
        continue

    job_record = discovered_jobs[idx]
    url        = job_record["url"]

    delay = random.uniform(2.0, 3.5)
    log.info(f"Sleeping {delay:.2f}s ...")
    time.sleep(delay)

    log.info(f"Inspecting #{idx}: {url}")

    try:
        response   = requests.get(url, headers=HEADERS, timeout=15)
        soup       = BeautifulSoup(response.text, "html.parser")
        script_tag = soup.find("script", type="application/ld+json")

        if not script_tag:
            results.append({"idx": idx, "url": url, "status": "NO_JSON_LD"})
            continue

        raw         = json.loads(script_tag.string)
        graph       = raw.get("@graph", [])
        job_posting = next(
            (node for node in graph if node.get("@type") == "JobPosting"),
            None
        )

        if not job_posting:
            results.append({"idx": idx, "url": url, "status": "NO_JOB_POSTING"})
            continue

        results.append({
            "idx":         idx,
            "status":      "OK",
            "title":       job_posting.get("title"),
            "datePosted":  job_posting.get("datePosted"),
            "employmentType": job_posting.get("employmentType"),
            # These are the nested ones we care most about
            "baseSalary":            "✅" if job_posting.get("baseSalary") else "❌",
            "jobLocation":           "✅" if job_posting.get("jobLocation") else "❌",
            "hiringOrganization":    "✅" if job_posting.get("hiringOrganization") else "❌",
            "jobBenefits":           "✅" if job_posting.get("jobBenefits") else "❌",
            "industry":              "✅" if job_posting.get("industry") else "❌",
            "validThrough":          "✅" if job_posting.get("validThrough") else "❌",
            "applicantLocationRequirements": "✅" if job_posting.get("applicantLocationRequirements") else "❌",
            "directApply":           job_posting.get("directApply"),
            # Drill into nested fields we'll need to flatten in dbt
            "salary_currency":       job_posting.get("baseSalary", {}).get("currency"),
            "salary_min":            job_posting.get("baseSalary", {}).get("value", {}).get("minValue"),
            "salary_max":            job_posting.get("baseSalary", {}).get("value", {}).get("maxValue"),
            "company_name":          job_posting.get("hiringOrganization", {}).get("name"),
            "location_city":         job_posting.get("jobLocation", {}).get("address", {}).get("addressLocality"),
            "location_state":        job_posting.get("jobLocation", {}).get("address", {}).get("addressRegion"),
        })

    except Exception as e:
        results.append({"idx": idx, "url": url, "status": f"ERROR: {e}"})

# ------------------------------------------------------------------ #
# Print summary table
# ------------------------------------------------------------------ #
print(f"\n{'#':<5} {'Status':<8} {'baseSalary':<12} {'jobLocation':<12} "
      f"{'hiringOrg':<10} {'benefits':<10} {'industry':<10} {'Title'}")
print("-" * 110)
for r in results:
    if r["status"] != "OK":
        print(f"{r['idx']:<5} {r['status']:<8}")
        continue
    print(
        f"{r['idx']:<5} {r['status']:<8} "
        f"{r['baseSalary']:<12} {r['jobLocation']:<12} "
        f"{r['hiringOrganization']:<10} {r['jobBenefits']:<10} "
        f"{r['industry']:<10} {r['title'][:50]}"
    )

print()
print("--- Nested field drill-down (only where baseSalary present) ---")
print(f"{'#':<5} {'Company':<35} {'City':<20} {'State':<8} "
      f"{'Currency':<10} {'Min Salary':<12} {'Max Salary'}")
print("-" * 110)
for r in results:
    if r["status"] != "OK":
        continue
    print(
        f"{r['idx']:<5} {str(r['company_name']):<35} "
        f"{str(r['location_city']):<20} {str(r['location_state']):<8} "
        f"{str(r['salary_currency']):<10} {str(r['salary_min']):<12} "
        f"{str(r['salary_max'])}"
    )

2026-05-24 18:01:53,015 [INFO] Sleeping 3.25s ...
2026-05-24 18:01:56,273 [INFO] Inspecting #0: https://www.builtinnyc.com/job/data-analyst-data-cloud-intelligence/9422585
2026-05-24 18:01:56,957 [INFO] Sleeping 2.00s ...
2026-05-24 18:01:58,963 [INFO] Inspecting #5: https://www.builtinnyc.com/job/data-analyst-investment-products-contract/9438769
2026-05-24 18:02:00,139 [INFO] Sleeping 3.12s ...
2026-05-24 18:02:03,262 [INFO] Inspecting #10: https://www.builtinnyc.com/job/afc-systems-data-analyst/9287838
2026-05-24 18:02:03,905 [INFO] Sleeping 2.62s ...
2026-05-24 18:02:06,532 [INFO] Inspecting #20: https://www.builtinnyc.com/job/market-data-analyst/9217412
2026-05-24 18:02:07,290 [INFO] Sleeping 3.21s ...
2026-05-24 18:02:10,503 [INFO] Inspecting #30: https://www.builtinnyc.com/job/policy-and-data-analyst/9318400
2026-05-24 18:02:11,070 [INFO] Sleeping 2.41s ...
2026-05-24 18:02:13,481 [INFO] Inspecting #40: https://www.builtinnyc.com/job/data-analytics-reporting-analyst-investment-co


#     Status   baseSalary   jobLocation  hiringOrg  benefits   industry   Title
--------------------------------------------------------------------------------------------------------------
0     OK       ✅            ✅            ✅          ✅          ✅          Data Analyst, Data Cloud Intelligence
5     OK       ✅            ✅            ✅          ✅          ✅          Data Analyst - Investment products (CONTRACT)
10    OK       ✅            ✅            ✅          ✅          ✅          AFC Systems Data Analyst
20    OK       ✅            ✅            ✅          ✅          ✅          Market Data Analyst
30    OK       ❌            ✅            ✅          ✅          ✅          Policy and Data Analyst
40    OK       ✅            ✅            ✅          ✅          ✅          Data, Analytics & Reporting Analyst -  Investment 
45    OK       ✅            ✅            ✅          ✅          ✅          Technology Program Quality Analyst (Automation & D

--- Nested field drill-down (only 

In [7]:
if SCRAPE_CACHE.exists():
    print("Scrape cache exists — loading from file.")
    with SCRAPE_CACHE.open() as f:
        scraped_jobs = json.load(f)
else:
    scraped_jobs: list[dict] = []
    failed_urls:  list[str]  = []

    log.info(f"Starting Stage 2 scrape — {len(discovered_jobs)} jobs to process...")

    for i, job_record in enumerate(discovered_jobs):
        url   = job_record["url"]
        delay = random.uniform(MIN_SCRAPE_DELAY, MAX_SCRAPE_DELAY)

        log.info(f"[{i+1}/{len(discovered_jobs)}] Sleeping {delay:.2f}s ...")
        time.sleep(delay)
        log.info(f"[{i+1}/{len(discovered_jobs)}] Scraping: {url}")

        try:
            response = requests.get(url, headers=HEADERS, timeout=15)

            if response.status_code != 200:
                log.warning(f"  HTTP {response.status_code} — skipping.")
                failed_urls.append(url)
                continue

            soup       = BeautifulSoup(response.text, "html.parser")
            script_tag = soup.find("script", type="application/ld+json")

            if not script_tag:
                log.warning(f"  No JSON-LD found — skipping.")
                failed_urls.append(url)
                continue

            raw         = json.loads(script_tag.string)
            graph       = raw.get("@graph", [])
            job_posting = next(
                (node for node in graph if node.get("@type") == "JobPosting"),
                None
            )

            if not job_posting:
                log.warning(f"  No JobPosting node — skipping.")
                failed_urls.append(url)
                continue

            scraped_jobs.append({
                # Scraper metadata
                "source_url":  url,
                "crawl_title": job_record["title"],
                "scraped_at":  datetime.now(timezone.utc).isoformat(),
                # Full raw JobPosting node — untouched, dbt flattens downstream
                "job_posting": job_posting,
            })

            log.info(f"  ✓ {job_posting.get('title', 'N/A')}")

        except Exception as e:
            log.error(f"  Error scraping {url}: {e}", exc_info=True)
            failed_urls.append(url)

    with SCRAPE_CACHE.open("w") as f:
        json.dump(scraped_jobs, f, indent=2)

    log.info(
        f"Scrape complete — {len(scraped_jobs)} succeeded, "
        f"{len(failed_urls)} failed."
    )

    if failed_urls:
        print(f"\n⚠️  Failed URLs ({len(failed_urls)}):")
        for u in failed_urls:
            print(f"  {u}")

print(f"\nTotal scraped: {len(scraped_jobs)}")

2026-05-24 18:04:10,969 [INFO] Starting Stage 2 scrape — 46 jobs to process...
2026-05-24 18:04:10,971 [INFO] [1/46] Sleeping 2.21s ...
2026-05-24 18:04:13,184 [INFO] [1/46] Scraping: https://www.builtinnyc.com/job/data-analyst-data-cloud-intelligence/9422585
2026-05-24 18:04:13,814 [INFO]   ✓ Data Analyst, Data Cloud Intelligence
2026-05-24 18:04:13,814 [INFO] [2/46] Sleeping 2.36s ...
2026-05-24 18:04:16,180 [INFO] [2/46] Scraping: https://www.builtinnyc.com/job/business-intelligence-data-analyst/9246907
2026-05-24 18:04:17,027 [INFO]   ✓ Business Intelligence Data Analyst
2026-05-24 18:04:17,028 [INFO] [3/46] Sleeping 2.40s ...
2026-05-24 18:04:19,437 [INFO] [3/46] Scraping: https://www.builtinnyc.com/job/data-governance-analyst/9224839
2026-05-24 18:04:20,042 [INFO]   ✓ Data Governance Analyst
2026-05-24 18:04:20,042 [INFO] [4/46] Sleeping 2.08s ...
2026-05-24 18:04:22,130 [INFO] [4/46] Scraping: https://www.builtinnyc.com/job/private-markets-data-analyst-senior-associate/9177772
2


Total scraped: 46


In [8]:
from collections import Counter

total        = len(scraped_jobs)
field_counts = Counter()
null_counts  = Counter()

for record in scraped_jobs:
    jp = record.get("job_posting", {})
    field_counts.update(jp.keys())
    for k, v in jp.items():
        if v is None or v == "" or v == []:
            null_counts[k] += 1

print(f"Total scraped jobs: {total}")
print()
print(f"{'Field':<40} {'Present':>7}  {'Null/empty':>10}  {'% populated'}")
print("-" * 75)
for field, count in sorted(field_counts.items(), key=lambda x: -x[1]):
    nulls    = null_counts.get(field, 0)
    filled   = count - nulls
    pct      = (filled / total * 100) if total else 0
    bar      = "█" * int(pct / 5)
    print(f"  {field:<38} {count:>7}  {nulls:>10}  {pct:>6.0f}%  {bar}")

print()

# Drill into baseSalary.value.unitText to catch hourly vs annual
print("--- baseSalary.value.unitText distribution ---")
unit_counts: Counter = Counter()
for record in scraped_jobs:
    jp         = record.get("job_posting", {})
    base_sal   = jp.get("baseSalary", {})
    unit_text  = base_sal.get("value", {}).get("unitText") if base_sal else None
    unit_counts[unit_text or "missing"] += 1

for unit, count in unit_counts.most_common():
    print(f"  {unit:<20} {count} jobs")

Total scraped jobs: 46

Field                                    Present  Null/empty  % populated
---------------------------------------------------------------------------
  @context                                    46           0     100%  ████████████████████
  @type                                       46           0     100%  ████████████████████
  title                                       46           0     100%  ████████████████████
  description                                 46           0     100%  ████████████████████
  directApply                                 46           0     100%  ████████████████████
  identifier                                  46           0     100%  ████████████████████
  datePosted                                  46           0     100%  ████████████████████
  employmentType                              46           0     100%  ████████████████████
  applicantLocationRequirements               46           0     100%  ███████████████████

In [9]:
ingested_at = datetime.now(timezone.utc).isoformat()

snowflake_rows = [
    {
        "SOURCE":      "builtin",
        "RAW_PAYLOAD": {
            # Scraper metadata
            "source_url":  record["source_url"],
            "crawl_title": record["crawl_title"],
            "scraped_at":  record["scraped_at"],
            # Full raw JobPosting node spread in — all nested dicts intact
            # dbt will flatten: RAW_PAYLOAD:hiringOrganization:name::string etc.
            **record["job_posting"],
        },
        "INGESTED_AT": ingested_at,
    }
    for record in scraped_jobs
]

print(f"Rows ready for Snowflake: {len(snowflake_rows)}")
print()

# Show the full key set that will land in RAW_PAYLOAD
sample_payload = snowflake_rows[0]["RAW_PAYLOAD"]
print(f"RAW_PAYLOAD keys ({len(sample_payload)}):")
for k in sample_payload.keys():
    v = sample_payload[k]
    if isinstance(v, dict):
        print(f"  {k:<35} [dict]  subkeys: {list(v.keys())}")
    elif isinstance(v, list):
        print(f"  {k:<35} [list, len={len(v)}]")
    else:
        display = str(v)[:80] + "..." if len(str(v)) > 80 else str(v)
        print(f"  {k:<35} {display}")

print()
print("Sample Snowflake row (nested fields shown as-is):")
print(json.dumps(
    {
        "SOURCE":      snowflake_rows[0]["SOURCE"],
        "INGESTED_AT": snowflake_rows[0]["INGESTED_AT"],
        "RAW_PAYLOAD": {
            k: v for k, v in sample_payload.items()
            if k in [
                "title", "source_url", "crawl_title", "datePosted",
                "employmentType", "industry", "hiringOrganization",
                "jobLocation", "baseSalary", "validThrough",
            ]
        },
    },
    indent=2,
    default=str,
))

Rows ready for Snowflake: 46

RAW_PAYLOAD keys (19):
  source_url                          https://www.builtinnyc.com/job/data-analyst-data-cloud-intelligence/9422585
  crawl_title                         Data Analyst, Data Cloud Intelligence
  scraped_at                          2026-05-24T22:04:13.814159+00:00
  @context                            https://schema.org
  @type                               JobPosting
  title                               Data Analyst, Data Cloud Intelligence
  description                         <p><strong>WHO WE ARE </strong><span> </span><span>&nbsp;</span></p><p><span>Zet...
  directApply                         True
  jobLocationType                     TELECOMMUTE
  identifier                          [dict]  subkeys: ['@type', 'name', 'value']
  baseSalary                          [dict]  subkeys: ['@type', 'currency', 'value']
  datePosted                          2026-05-19
  employmentType                      FULL_TIME
  applicantLocationRequi